In [ ]:
import polars as pl 


In [ ]:
data = pl.read_csv("CleanedTextSocialDistanceFeb27_sentiment_conv_full.csv")

data.describe()


 Reindex dataframe by conversation IDs.

In [ ]:
conversation_data = data.group_by(["conversation_id"]).agg(pl.all())
# conversation_ID n | Inbound = 1 | ...
# conversation_ID n | Inbound = 0 | ...

#conversation_data.describe()
conversation_data.head()


 (1) Does Concrete answers take a longer time?

 Side note: exploding is suboptimal, but simple enough for our purposes since these conversations aren't going to be particularly large

In [ ]:
# (a) Time Difference until first support tweet after initial customer tweet
conversationsWithSupportResponses = conversation_data.filter(
    pl.col("inbound").list.eval(pl.element() == 0).list.any()
)

timeToFirstCustomerSupport = (conversationsWithSupportResponses
    .explode(["inbound", "created_at"])
    .with_columns(pl.col("created_at").str.strptime(pl.Datetime, "%a %b %d %H:%M:%S %z %Y"))
    .group_by("conversation_id")
    .agg((pl.col("created_at").filter(pl.col("inbound") == 0).min() - 
         pl.col("created_at").filter(pl.col("inbound") == 1).min())
         .dt.total_seconds()
         .alias("time_to_support"))
)




In [ ]:
print(len(conversation_data), len(conversationsWithSupportResponses))
len(conversationsWithSupportResponses) - len(conversation_data)


In [ ]:


averageTimeToFirstResponse = timeToFirstCustomerSupport.select(pl.col("time_to_support").mean())
#print(f"Average time to customer support response: {averageTimeToResponse} seconds")

timeToFirstCustomerSupport.describe()

# there's some bogus data still, min = -2.2427e8; max = 2.41937519e8 seconds?
# trim out 5th and 95th percentiles to see if that works. Check data tomorrow.


In [ ]:
p5 = timeToFirstCustomerSupport["time_to_support"].quantile(0.05)
p95 = timeToFirstCustomerSupport["time_to_support"].quantile(0.95)

print(f"5th percentile: {p5 / 60:.1f} minutes")
print(f"95th percentile: {p95 / 60:.1f} minutes")

timeToFirstCustomerSupport_trimmed = timeToFirstCustomerSupport.filter(
    (pl.col("time_to_support") >= p5) &
    (pl.col("time_to_support") <= p95)
)
timeToFirstCustomerSupport_trimmed.describe()


 Average duration per support tweet

In [ ]:
# not what you asked for: average time between support responses (time between first and last support response divided by number of support responses)
averageTimeBetweenSupportResponse = (conversationsWithSupportResponses
    .explode(["inbound", "created_at"])
    .with_columns(
        pl.col("created_at").str.strptime(pl.Datetime, "%a %b %d %H:%M:%S %z %Y")
    )
    .filter(pl.col("inbound") == 0)  
    .group_by("conversation_id")
    .agg([
        pl.col("created_at").min().alias("first_support"),
        pl.col("created_at").max().alias("last_support"),
        pl.col("created_at").count().alias("support_tweet_count")
    ])
    .filter(pl.col("support_tweet_count") > 1)  # Need at least 2 tweets
    .with_columns([
        # Time span / (number of tweets - 1) = average gap
        ((pl.col("last_support") - pl.col("first_support")).dt.total_seconds() / 
         (pl.col("support_tweet_count") - 1))
        .alias("avg_seconds_between_support_tweets")
    ])
)

print(averageTimeBetweenSupportResponse.select([
    "conversation_id",
    "support_tweet_count",
    "avg_seconds_between_support_tweets"
]).head())


In [ ]:
p5 = averageTimeBetweenSupportResponse["avg_seconds_between_support_tweets"].quantile(0.05)
p95 = averageTimeBetweenSupportResponse["avg_seconds_between_support_tweets"].quantile(0.95)

avg_trimmed = averageTimeBetweenSupportResponse.filter(
    (pl.col("avg_seconds_between_support_tweets") >= p5) &
    (pl.col("avg_seconds_between_support_tweets") <= p95)
)

print("Trimmed statistics (middle 90%):")
print(avg_trimmed["avg_seconds_between_support_tweets"].describe())


In [ ]:
averageTimePerTweet = (conversationsWithSupportResponses
    .explode(["inbound", "created_at"])
    .with_columns(
        pl.col("created_at").str.strptime(pl.Datetime, "%a %b %d %H:%M:%S %z %Y")
    )
    .group_by("conversation_id")
    .agg([
        pl.col("created_at").min().alias("first_tweet"),
        pl.col("created_at").max().alias("last_tweet"),
        pl.col("created_at").count().alias("total_tweets")
    ])
    .with_columns([
        ((pl.col("last_tweet") - pl.col("first_tweet")).dt.total_seconds() / 
         pl.col("total_tweets"))
        .alias("avg_seconds_between_all_tweets")
    ])
)  

print(averageTimePerTweet.select([
    "conversation_id",
    "total_tweets",
    "avg_seconds_between_all_tweets"
]).head())  

averageTimePerTweet.describe()


In [ ]:
p5 = averageTimePerTweet["avg_seconds_between_all_tweets"].quantile(0.05)
p95 = averageTimePerTweet["avg_seconds_between_all_tweets"].quantile(0.95)

avg_trimmed = averageTimePerTweet.filter(
    (pl.col("avg_seconds_between_all_tweets") >= p5) &
    (pl.col("avg_seconds_between_all_tweets") <= p95)
)

print("Trimmed statistics (middle 90%):")
print(avg_trimmed["avg_seconds_between_all_tweets"].describe())


 Combining tables with original data

In [ ]:
conversation_data_with_time_stats = (conversation_data
    # Join time to first support response
    .join(
        timeToFirstCustomerSupport_trimmed.select([
            "conversation_id",
            "time_to_support"
        ]),
        on="conversation_id",
        how="left"
    )
    # Join average time between all tweets
    .join(
        avg_trimmed.select([
            "conversation_id",
            "avg_seconds_between_all_tweets",
            "total_tweets"
        ]),
        on="conversation_id",
        how="left"
    )
    # Join average time between support tweets
    #.join(
    #    averageTimeBetweenSupportResponse.select([
    #        "conversation_id",
    #        "avg_seconds_between_support_tweets"
    #    ]),
    #    on="conversation_id",
    #    how="left"
    #)
)

# cull conversations that data wasn't collected on (appeared on 5th or 95th percentile on one of them)
conversation_data_with_time_stats_trimmed = conversation_data_with_time_stats.filter(
    pl.col("time_to_support").is_not_null() &
    pl.col("avg_seconds_between_all_tweets").is_not_null()
)   


In [ ]:
conversation_data_with_time_stats_trimmed.describe()
conversation_data_with_time_stats_trimmed.head()


 Plots for (1)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# average concreteness of support tweets per conversation
support_social_distance = (conversation_data_with_time_stats_trimmed
    .explode(["inbound", "social_distance_score"])
    .filter(pl.col("inbound") == 0)
    .group_by("conversation_id")
    .agg(pl.col("social_distance_score").mean().alias("avg_social_distance_score_support"))
)

concreteness_data = conversation_data_with_time_stats_trimmed.join(
    support_social_distance.select([
        "conversation_id",
        "avg_social_distance_score_support"
    ]),
    on="conversation_id",
    how="left"
)

concreteness_data.describe()
concreteness_data.head()


In [ ]:
# concreteness of support tweets over time to first support response
x = concreteness_data.select("time_to_support").to_series()
y = concreteness_data.select("avg_social_distance_score_support").to_series()

plt.figure(figsize=(10, 6))
sns.scatterplot(x=x, y=y, alpha=0.6)
plt.title('Social Distance vs Time to First Support Response')
plt.show()

plt.savefig("social_distance_vs_time_to_support.png")
# weird plot.


In [ ]:
# concreteness of support tweets over average time between all tweets

x = concreteness_data.select("avg_seconds_between_all_tweets").to_series()
y = concreteness_data.select("avg_social_distance_score_support").to_series()

plt.figure(figsize=(10, 6))
sns.scatterplot(x=x, y=y, alpha=0.6)
plt.title('Social Distance vs Time Between All Tweets')
plt.xlabel('Average Time Between All Tweets')
plt.ylabel('Average Social Distance Score (Support)')
plt.show()

plt.savefig("social_distance_vs_time_between_all_tweets.png")


In [ ]:
# average customer sentiment per conversation
average_customer_sentiment = (conversation_data_with_time_stats_trimmed
    .explode(["inbound", "sentiment_positive_prob"])
    .filter(pl.col("inbound") == 1)
    .group_by("conversation_id")
    .agg(pl.col("sentiment_positive_prob").mean().alias("avg_sentiment_positive_prob"))
)

sentiment_data = conversation_data_with_time_stats_trimmed.join(
    average_customer_sentiment.select([
        "conversation_id",
        "avg_sentiment_positive_prob"
    ]),
    on="conversation_id",
    how="left"
)

sentiment_data.describe()
sentiment_data.head()

plt.savefig("sentiment_vs_time_to_support.png")


In [ ]:
x = sentiment_data.select("time_to_support").to_series()
y = sentiment_data.select("avg_sentiment_positive_prob").to_series()

plt.figure(figsize=(10, 6))
sns.scatterplot(x=x, y=y, alpha=0.6)
plt.title('Customer Sentiment vs Time to First Support Response')
plt.show()


In [ ]:
x = sentiment_data.select("avg_seconds_between_all_tweets").to_series()
y = sentiment_data.select("avg_sentiment_positive_prob").to_series()

plt.figure(figsize=(10, 6))
sns.scatterplot(x=x, y=y, alpha=0.6)
plt.title('Customer Sentiment vs Time betweween All Tweets')
plt.show()

plt.savefig("sentiment_vs_time_between_all_tweets.png")


In [ ]:
# More concrete answers means more satisfaction?

concrete_sentiment_data = (conversation_data_with_time_stats_trimmed
    .join(
        support_social_distance.select([
            "conversation_id",  
            "avg_social_distance_score_support"
        ]),
        on="conversation_id",
        how="left"
    )
    .join(
        average_customer_sentiment.select([
            "conversation_id",
            "avg_sentiment_positive_prob"
        ]),
        on="conversation_id",
        how="left"
    )
)

x = concrete_sentiment_data.select("avg_social_distance_score_support").to_series()
y = concrete_sentiment_data.select("avg_sentiment_positive_prob").to_series()

plt.figure(figsize=(10, 6))
sns.scatterplot(x=x, y=y, alpha=0.6)
plt.title('Customer Sentiment vs Support Social Distance')
plt.xlabel('Average Social Distance Score (Support)')
plt.ylabel('Average Sentiment Positive Probability (Customer)')
plt.show()

plt.savefig("sentiment_vs_social_distance.png")


